# Intake: <new dataset>

Copy this folder to `datasets/<name>/`, rename the notebook, and fill in the
marked cells. Nothing else in the hub needs to change -- `build_run.ipynb`
discovers any folder with an `inputs/config.json`.

## What a bundle has to provide

One row per reaction or well, with these columns:

| column | what it is |
| --- | --- |
| `row_id` | `"reaction:1"`, `"reaction:2"`, ... in table order (`prep.add_row_ids`) |
| *group* | the varied component's name -- ligand, catalyst, whatever this screen varies |
| `kraken_id` | that component's Kraken id, the join key for descriptors |
| *target* | the numeric response |
| *conditions* | one column per shared categorical field |

Any other column is carried through and ignored by the models, so keep plate
and well ids if you have them.

## The two constraints worth knowing before you start

1. **Every group needs a Kraken id.** The descriptor sections (`selected_2`,
   `selected_5`, `pc_top`, `pc_scores`) are Kraken descriptors joined on it. A
   bidentate ligand or a component that is not in Kraken has no row, so it has
   to be excluded here -- and the exclusion recorded.
2. **The reference PCA is never refitted.** `prep.write_bundle` copies it, so
   PC1..PC4 mean the same axes in every dataset. That is what makes `pc_top`
   and `pc_scores` comparable across screens.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()            # datasets/<name>/
ROOT = HERE.parent.parent              # gp_collab_hub/
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(HERE))

from gpc import prep
from gpc.data import load_bundle

pd.set_option("display.width", 220, "display.max_columns", 80)

OUT = HERE / "inputs"                  # the bundle this notebook writes

# ---- Editable: fill these in --------------------------------------------
DATASET      = "newdataset"          # folder name, lower case
DISPLAY_NAME = "New Dataset"         # what appears on figure titles
TARGET       = "yield"               # the numeric response column
GROUP        = "ligand"              # the varied component's column
CATEGORICAL  = ["base", "solvent"]   # the shared condition fields
SOURCE_URL   = ""                    # where the data came from
RAW          = HERE / "raw"
# --------------------------------------------------------------------------

## 1. Build the reactions table

**This is the cell to write.** Load the screen's own file and reshape it to the
schema above. Whatever you drop, record why -- put it in `audit` and it is
written into the bundle alongside the data.

The stub below makes a tiny fake frame so the rest of the notebook runs end to
end before you have real data in place. Replace it.

In [ ]:
audit = {"source": SOURCE_URL, "dropped": {}}

# ---- Replace everything between here --------------------------------------
rng = np.random.default_rng(0)
levels = {"ligand": ["XPhos", "SPhos", "P(tBu)3", "RuPhos"],
          "base": ["K3PO4", "Cs2CO3"], "solvent": ["MeCN", "THF", "Toluene"]}
grid = pd.MultiIndex.from_product(list(levels.values()),
                                  names=list(levels)).to_frame(index=False)
grid[TARGET] = rng.uniform(0, 100, len(grid)).round(2)
reactions = grid
# ---- and here -------------------------------------------------------------

reactions = prep.add_row_ids(reactions)
print(f"{len(reactions)} rows x {reactions.shape[1]} columns")
display(reactions.head(3))

## 2. The mapping

One Kraken id per group value. Kraken's own spellings often differ from the
paper's, so look them up rather than assume:

```python
prep.find_kraken_id("BrettPhos")      # every Kraken row whose name contains it
```

`attach_kraken` fails loudly on any group value you have not mapped -- a silent
NaN here becomes a missing descriptor hundreds of lines further on.

In [ ]:
# ---- Editable -----------------------------------------------------------
MAPPING = {"XPhos": 1, "SPhos": 3, "P(tBu)3": 8, "RuPhos": 4}
# --------------------------------------------------------------------------

display(prep.find_kraken_id("BrettPhos").head())

reactions = prep.attach_kraken(reactions, GROUP, MAPPING)
identifiers = prep.kraken_identifiers().set_index("id")
display(pd.DataFrame([{GROUP: name, "kraken_id": kid,
                       "kraken_name": identifiers["ligand"].get(kid, "?")}
                      for name, kid in MAPPING.items()]))

## 4. Describe it, then validate

The table below is the record of what this bundle claims about itself: rows and
target statistics per group. Read it before writing -- an unbalanced design
shows up here, and it changes how the LOLO folds should be read (their sizes
follow these counts).

In [ ]:
display(prep.describe(reactions, {"data": {"target": TARGET, "group": GROUP}}))

In [ ]:
# ---- Editable: what this dataset IS -------------------------------------
cfg = prep.bundle_config(
    dataset=DATASET,
    display_name=DISPLAY_NAME,
    target=TARGET,
    group=GROUP,
    categorical=CATEGORICAL,
    reactions=reactions,
    source=SOURCE_URL,
    # models/methods default sensibly: all five sections, and the method list
    # whose matched control has this screen's own group count. Pass explicit
    # lists here to override.
)
# --------------------------------------------------------------------------

display(pd.json_normalize(cfg["data"]).T.rename(columns={0: "value"}))
print("methods:", cfg["evaluation"]["methods"])

# Every check load_bundle will make, run here so a bad bundle fails at the
# point it was built rather than on the cluster three hours into a job.
display(prep.check_bundle(reactions, cfg, prep.kraken_features()))

## 5. Write the bundle

`write_bundle` writes `reactions.csv`, the Kraken descriptor rows for the
ligands this screen uses, `ligand_mapping.csv`, the copied reference PCA and
`config.json`. The PCA is **copied, never refitted** -- PC1..PC4 have to mean
the same axes in every dataset or `pc_top` and `pc_scores` stop being
comparable, which is the point of running them.

In [ ]:
bundle = prep.write_bundle(OUT, reactions, cfg, MAPPING, audit=audit)

# Read it straight back through the engine's own loader: if this succeeds, the
# bundle is usable by `gpc train` exactly as written.
back, ligands, reference, prepared = load_bundle(bundle)
print(f"\nreloaded: {len(back)} rows, {back[cfg['data']['group']].nunique()} "
      f"{cfg['data']['group']}s, {len(ligands)} descriptor rows, "
      f"PCA over {len(reference.columns)} descriptors")
display(pd.DataFrame([{"file": p.name, "KB": round(p.stat().st_size / 1024, 1)}
                      for p in sorted(bundle.iterdir())]))
print("\nNext: build_run.ipynb in the hub root, and pick this dataset.")